In [0]:
df_facilities = spark.sql(f"select * from regis_healthcare.silver.facilities;")
df_facilities.createOrReplaceTempView("facilities")

In [0]:
print(f"facilities = {df_facilities.columns}")

In [0]:
# dim_facilities --> Source: facilities

dim_facilities = spark.sql("""select * from facilities order by facility_id""")
# display(dim_facilities)
# #--------------------------
# from pyspark.sql.functions import col, date_format
# # Create date_key column in YYYYMMDD format
# dim_facilities = dim_facilities.withColumn("cr_at_date_key", date_format(col("created_at"), "yyyyMMdd"))
# # Optionally cast to integer for warehouse-style keys
# dim_facilities = dim_facilities.withColumn("cr_at_date_key", col("cr_at_date_key").cast("int"))
# #--------------------------
from pyspark.sql.functions import col, regexp_replace
dim_facilities = dim_facilities.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(dim_facilities)
dim_facilities = dim_facilities.select(
    "facility_key",
    "facility_id",
    "facility_name",
    "address",
    "suburb",
    "state",
    "postcode",
    "phone",
    "email",
    "capacity",
    "accreditation_status",
    "created_at")
display(dim_facilities)

#### cataloge 

In [0]:
# # cataloge 
# dim_facilities.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_facilities")

In [0]:
dim_facilities.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_facilities")
print(sb_dim_facilities.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_facilities")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_facilities")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.facility_key = source.facility_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_facilities;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_facilities;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# # gold load to s3
# dim_facilities.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_facilities")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_facilities"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = dim_facilities

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.facility_key = source.facility_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
